In [0]:
import pyspark.sql.functions as F

In [0]:
df_emp_a = (
    spark.read.table("ai_lab_demo.bronze.employees_src_a")
)
df_emp_a = df_emp_a.withColumn("Date_of_Birth", F.to_date("Date_of_Birth", "dd/MM/yyyy"))
df_emp_a.display()

In [0]:
df_emp_b = (
    spark.read.table("ai_lab_demo.bronze.employees_src_b")
)

In [0]:
df_emp_b =  df_emp_b.withColumn("Date_of_Birth", F.to_date("Date_of_Birth", "dd/MM/yyyy"))
df_emp_b.display()

In [0]:
df_emp_b_trans = (
    df_emp_b
    .withColumn("First_Name", F.get_json_object("customer", "$.First_Name"))
    .withColumn("Last_Name", F.get_json_object("customer", "$.Last_Name"))
    .select(
        "Employee_id",
        "First_Name",
        "Last_Name",
        "Gender",
        "Salary",
        "Date_of_Birth",
        "Age",
        "Country",
        "Department_id",
        "Date_of_Joining",
        "Manager_id",
        "Currency",
        "End_Date",
        "file_path",
        "ingestion_time"
    )
)


In [0]:
df_emp_c = (
    spark.read.table("ai_lab_demo.bronze.employees_src_c")
)

In [0]:
df_emp_trans_c = (
    df_emp_c
    .withColumn("First_Name", F.regexp_extract("customer", r"<First_Name>(.*?)</First_Name>", 1))
    .withColumn("Last_Name",F.regexp_extract("customer", r"<Last_Name>(.*?)</Last_Name>", 1))
    .withColumn("Date_of_Birth", F.to_date("Date_of_Birth", "dd/MM/yyyy"))
    .select(
        "Employee_id",
        "First_Name",
        "Last_Name",
        "Gender",
        "Salary",
        "Date_of_Birth",
        "Age",
        "Country",
        "Department_id",
        "Date_of_Joining",
        "Manager_id",
        "Currency",
        "End_Date",
        "file_path",
        "ingestion_time"
    )
)

In [0]:
df_emp_trans_c.display()

In [0]:
df_emp_src_a = (
spark.read.table("ai_lab_demo.bronze.employees_src_a")
)

df_emp_src_b = (
spark.read.table("ai_lab_demo.bronze.employees_src_b")
)

df_emp_src_c = (
spark.read.table("ai_lab_demo.bronze.employees_src_c")
)

max_ingestion_time_a = (
    df_emp_src_a
    .select(F.max("ingestion_time"))
    .first()[0]
)
max_ingestion_time_b = (
    df_emp_src_b
    .select(F.max("ingestion_time"))
    .first()[0]
)
max_ingestion_time_c = (
    df_emp_src_c
    .select(F.max("ingestion_time"))
    .first()[0]
)

max_ingestion_time = max(max_ingestion_time_a, max_ingestion_time_b, max_ingestion_time_c)

In [0]:
max_ingestion_time

In [0]:
unprocessed_df_emp_a, unprocessed_df_emp_b, unprocessed_df_emp_c = get_unprocessed_bronze_df(df_emp_src_a, df_emp_src_b, df_emp_src_c, max_ingestion_time)